In [1]:
!pip install scipy statsmodels scikit-learn plotly -q

import numpy as np
import pandas as pd

from scipy import stats
import statsmodels.api as sm

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

import plotly.express as px

print("All libraries imported successfully.")

All libraries imported successfully.


**Cell 2: Create a student dataset**

In [2]:
np.random.seed(42)

number_of_students = 100

df = pd.DataFrame({
    "study_hours": np.random.randint(1, 10, number_of_students),
    "attendance": np.random.randint(50, 101, number_of_students),
    "previous_score": np.random.randint(35, 91, number_of_students),
    "sleep_hours": np.random.randint(4, 9, number_of_students)
})

df["final_score"] = (
    df["study_hours"] * 3
    + df["attendance"] * 0.25
    + df["previous_score"] * 0.45
    + df["sleep_hours"] * 1.5
    + np.random.normal(0, 4, number_of_students)
)

df["final_score"] = df["final_score"].clip(0, 100).round(2)

df.head()

,study_hours,attendance,previous_score,sleep_hours,final_score
0,7,84,89,4,94.71
1,4,86,38,8,59.64
2,8,96,89,7,98.12
3,5,63,45,6,58.06
4,7,52,90,4,86.87


**Cell 3: Understand the dataset**

In [5]:
print("Dataset shape:", df.shape)

print("\nColumn information:")
df.info()

print("\nStatistical summary:")
display(df.describe())

print("\nMissing values:")
print(df.isnull().sum())

Dataset shape: (100, 5)

Column information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   study_hours     100 non-null    int64  
 1   attendance      100 non-null    int64  
 2   previous_score  100 non-null    int64  
 3   sleep_hours     100 non-null    int64  
 4   final_score     100 non-null    float64
dtypes: float64(1), int64(4)
memory usage: 4.0 KB

Statistical summary:


,study_hours,attendance,previous_score,sleep_hours,final_score
count,100.000000,100.000000,100.000000,100.00000,100.000000
mean,5.320000,75.730000,61.970000,6.01000,72.240900
std,2.639559,15.208886,16.305452,1.43896,13.557082
min,1.000000,50.000000,35.000000,4.00000,39.640000
25%,3.000000,62.750000,50.000000,5.00000,62.675000
50%,5.000000,77.000000,62.500000,6.00000,72.460000
75%,8.000000,88.000000,74.000000,7.00000,81.265000
max,9.000000,100.000000,90.000000,8.00000,100.000000



Missing values:
study_hours       0
attendance        0
previous_score    0
sleep_hours       0
final_score       0
dtype: int64


**Cell 4: SciPy Pearson correlation**

In [9]:
correlation, p_value = stats.pearsonr(
    df["study_hours"],
    df["final_score"]
)

print("Correlation:", round(correlation, 3))
print("P-value:", round(p_value, 5))

if correlation > 0:
    print("Study hours and final score have a positive relationship.")
else:
    print("Study hours and final score have a negative relationship.")

if p_value < 0.05:
    print("The relationship is statistically significant.")
else:
    print("The relationship is not statistically significant.")

Correlation: 0.683
P-value: 0.0
Study hours and final score have a positive relationship.
The relationship is statistically significant.


**Cell 5: SciPy independent t-test**

In [10]:
attendance_median = df["attendance"].median()

high_attendance = df[
    df["attendance"] >= attendance_median
]["final_score"]

low_attendance = df[
    df["attendance"] < attendance_median
]["final_score"]

t_statistic, p_value = stats.ttest_ind(
    high_attendance,
    low_attendance,
    equal_var=False
)

print("High-attendance average:", round(high_attendance.mean(), 2))
print("Low-attendance average:", round(low_attendance.mean(), 2))
print("T-statistic:", round(t_statistic, 3))
print("P-value:", round(p_value, 5))

if p_value < 0.05:
    print("The two groups have a statistically significant difference.")
else:
    print("No statistically significant difference was found.")

High-attendance average: 75.13
Low-attendance average: 68.85
T-statistic: 2.345
P-value: 0.02117
The two groups have a statistically significant difference.


**Cell 6: Statsmodels regression**

In [11]:
X = df[
    [
        "study_hours",
        "attendance",
        "previous_score",
        "sleep_hours"
    ]
]

y = df["final_score"]

X_with_constant = sm.add_constant(X)

stats_model = sm.OLS(y, X_with_constant).fit()

print(stats_model.summary())

                            OLS Regression Results                            
Dep. Variable:            final_score   R-squared:                       0.921
Model:                            OLS   Adj. R-squared:                  0.918
Method:                 Least Squares   F-statistic:                     276.8
Date:                Sun, 12 Jul 2026   Prob (F-statistic):           1.97e-51
Time:                        12:44:38   Log-Likelihood:                -275.18
No. Observations:                 100   AIC:                             560.4
Df Residuals:                      95   BIC:                             573.4
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
const              1.2644      2.929      0.

**Cell 7: Scikit-learn model training**

In [12]:
X = df[
    [
        "study_hours",
        "attendance",
        "previous_score",
        "sleep_hours"
    ]
]

y = df["final_score"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

model = LinearRegression()

model.fit(X_train, y_train)

predictions = model.predict(X_test)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print("Model training completed.")

Training records: 80
Testing records: 20
Model training completed.


**Cell 8: Evaluate the model**

In [13]:
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print("Mean Absolute Error:", round(mae, 2))
print("R² Score:", round(r2, 3))

Mean Absolute Error: 2.74
R² Score: 0.926


**Cell 9: Compare actual and predicted scores**

In [14]:
results = pd.DataFrame({
    "Actual Score": y_test.values,
    "Predicted Score": predictions.round(2)
})

results["Error"] = (
    results["Actual Score"] - results["Predicted Score"]
).abs().round(2)

results.head(10)

,Actual Score,Predicted Score,Error
0,66.68,70.90,4.22
1,70.19,76.28,6.09
2,82.38,84.15,1.77
3,72.79,69.32,3.47
4,87.51,86.42,1.09
5,83.24,78.13,5.11
6,73.69,75.07,1.38
7,70.60,73.37,2.77
8,81.13,83.38,2.25
9,94.71,90.09,4.62


**Cell 10: Predict a new student’s score**

In [15]:
new_student = pd.DataFrame({
    "study_hours": [7],
    "attendance": [88],
    "previous_score": [72],
    "sleep_hours": [7]
})

predicted_score = model.predict(new_student)

print(
    "Predicted final score:",
    round(predicted_score[0], 2)
)

Predicted final score: 86.06


**Cell 11: Plotly interactive chart**

In [ ]:
fig = px.scatter(
    df,
    x="study_hours",
    y="final_score",
    color="attendance",
    size="previous_score",
    hover_data=["sleep_hours"],
    title="Study Hours vs Final Score"
)

fig.show()

**Cell 12: Feature coefficients**

In [ ]:
coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_
})

coefficients["Coefficient"] = coefficients["Coefficient"].round(3)

coefficients